In [79]:
import sys
import os
from dotenv import load_dotenv

# 1. Dapatkan path absolut dari direktori saat ini (ml_models)
current_dir = os.path.abspath('')

# 2. Dapatkan path dari parent directory (root folder)
parent_dir = os.path.dirname(current_dir)

# 3. Tambahkan parent directory ke sys.path agar Python bisa menemukan config.py
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

# 4. Load file .env secara eksplisit dari root folder agar DB_USER dkk terbaca
load_dotenv(os.path.join(parent_dir, '.env'))

True

In [80]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import optuna

from config import engine

In [81]:
def fetch_data():
    print("Fetching data from database...")
    query = """
        SELECT 
            d.date, 
            d.days_name, 
            d.month, 
            d.year, 
            d.is_weekend, 
            d.is_payday, 
            d.is_ramadhan, 
            pr.product_model, 
            pr.product_color AS color, 
            pr.product_size AS size, 
            pl.platform_name AS platform, 
            pm.payment_method_name AS payment_method, 
            l.province, 
            l.city, 
            f.quantity, 
            f.price, 
            f.discount, 
            f.total_amount
        FROM order_fact f
        LEFT JOIN date_dimension d ON f.date_id = d.date_id
        LEFT JOIN product_dimension pr ON f.product_id = pr.product_id
        LEFT JOIN platform_dimension pl ON f.platform_id = pl.platform_id
        LEFT JOIN payment_method_dimension pm ON f.payment_method_id = pm.payment_method_id
        LEFT JOIN location_dimension l ON f.location_id = l.location_id
        ORDER BY d.date DESC;
    """ 
    df = pd.read_sql(query, engine)
    print(f"Data loaded successfully! Shape: {df.shape}")
    return df

df_raw = fetch_data()

Fetching data from database...
2026-05-09 11:50:01,028 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-09 11:50:01,039 INFO sqlalchemy.engine.Engine DESCRIBE `janus_dw`.`
        SELECT 
            d.date, 
            d.days_name, 
            d.month, 
            d.year, 
            d.is_weekend, 
            d.is_payday, 
            d.is_ramadhan, 
            pr.product_model, 
            pr.product_color AS color, 
            pr.product_size AS size, 
            pl.platform_name AS platform, 
            pm.payment_method_name AS payment_method, 
            l.province, 
            l.city, 
            f.quantity, 
            f.price, 
            f.discount, 
            f.total_amount
        FROM order_fact f
        LEFT JOIN date_dimension d ON f.date_id = d.date_id
        LEFT JOIN product_dimension pr ON f.product_id = pr.product_id
        LEFT JOIN platform_dimension pl ON f.platform_id = pl.platform_id
        LEFT JOIN payment_method_dimension pm ON f.p

In [82]:
# Cell 2: Preprocessing & SKU Definition
df = df_raw.copy()

# Ensure datetime format
df['date'] = pd.to_datetime(df['date'])

df['sku'] = df['product_model'].astype(str) + "_" + \
            df['color'].astype(str) + "_" + \
            df['size'].astype(str)

# Sort by date for safety
df = df.sort_values(by='date').reset_index(drop=True)

print(f"Total Unique SKUs: {df['sku'].nunique()}")

Total Unique SKUs: 1017


In [83]:
# 1. Aggregate per-SKU
sku_features = df.groupby('sku').agg(
    freq=('quantity', 'count'),
    total_qty=('quantity', 'sum'),
    median_qty=('quantity', 'median'),
    mean_price=('price', 'mean'),
    total_revenue=('total_amount', 'sum'),
    std_qty=('quantity', 'std')
).fillna(0) # Fill NaN for std_qty if freq is 1

# Coefficient of Variation (CV)
sku_features['cv_qty'] = np.where(sku_features['total_qty'] > 0, 
                                  sku_features['std_qty'] / (sku_features['total_qty']/sku_features['freq']), 0)

# 2. Log Transform for skewed variables (Winsorize via RobustScaler later)
skewed_cols = ['total_qty', 'total_revenue', 'mean_price']
for col in skewed_cols:
    sku_features[f'log_{col}'] = np.log1p(sku_features[col])

# Drop original skewed columns
features_to_scale = [c for c in sku_features.columns if c not in skewed_cols]
X = sku_features[features_to_scale]

# 3. Scaling (RobustScaler handles outliers well)
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# Optional: PCA to reduce noise before K-Means
pca = PCA(n_components=0.95) # Keep 95% variance
X_pca = pca.fit_transform(X_scaled)

# 4. Find Best K using Silhouette & CH Score
best_k = 3
best_score = -1
metrics = []

for k in range(2, 8):
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = kmeans.fit_predict(X_pca)
    sil_score = silhouette_score(X_pca, labels)
    ch_score = calinski_harabasz_score(X_pca, labels)
    metrics.append({'k': k, 'silhouette': sil_score, 'ch_score': ch_score})
    
    if sil_score > best_score:
        best_score = sil_score
        best_k = k

print(pd.DataFrame(metrics))
print(f"\nSelected Optimal k: {best_k}")

# Final Clustering
final_kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=42)
sku_features['cluster'] = final_kmeans.fit_predict(X_pca)

# Map clusters back to main dataframe
df = df.merge(sku_features[['cluster']], on='sku', how='left')

   k  silhouette     ch_score
0  2    0.716553   914.066510
1  3    0.632778   995.028491
2  4    0.463592  1229.223027
3  5    0.376594  1244.990880
4  6    0.396600  1367.753267
5  7    0.410301  1486.734934

Selected Optimal k: 2


In [84]:
# Cell 4: Weekly Aggregation & Feature Engineering
# Agregasi data ke level Weekly per SKU
# 1. Agregasi dasar ke weekly (hanya untuk minggu yang ada transaksi)
weekly_df_raw = df.groupby(['sku', 'cluster', pd.Grouper(key='date', freq='W-MON')]).agg(
    weekly_qty=('quantity', 'sum'),
    avg_price=('price', 'mean'),
    avg_discount=('discount', 'mean'),
    is_payday=('is_payday', 'max'),
    is_ramadhan=('is_ramadhan', 'max')
).reset_index()

# 2. BIKIN DATA BERUNTUN (Zero-Filling)
# Cari rentang tanggal global (dari transaksi pertama hingga terakhir di seluruh data)
min_date = weekly_df_raw['date'].min()
max_date = weekly_df_raw['date'].max()
all_weeks = pd.date_range(start=min_date, end=max_date, freq='W-MON')

# Buat kombinasi semua SKU unik dengan semua minggu (Cartesian Product)
unique_skus = weekly_df_raw['sku'].unique()
multi_idx = pd.MultiIndex.from_product([unique_skus, all_weeks], names=['sku', 'date'])
full_grid = pd.DataFrame(index=multi_idx).reset_index()

# Gabungkan grid kosong dengan data agregasi raw
weekly_df = pd.merge(full_grid, weekly_df_raw, on=['sku', 'date'], how='left')

# 3. Handling Nilai Kosong (Imputation)
# Jika tidak ada transaksi, qty = 0
weekly_df['weekly_qty'] = weekly_df['weekly_qty'].fillna(0)
# Jika tidak ada transaksi, diskon & event = 0
weekly_df['avg_discount'] = weekly_df['avg_discount'].fillna(0)
weekly_df['is_payday'] = weekly_df['is_payday'].fillna(0)
weekly_df['is_ramadhan'] = weekly_df['is_ramadhan'].fillna(0)

# Untuk harga, kita gunakan Forward-Fill lalu Backward-Fill. 
# Logikanya: Jika minggu ini tidak ada penjualan, harganya diasumsikan sama dengan minggu sebelumnya.
weekly_df['avg_price'] = weekly_df.groupby('sku')['avg_price'].transform(lambda x: x.ffill().bfill())

# Kembalikan mapping cluster yang hilang akibat merge
cluster_map = df[['sku', 'cluster']].drop_duplicates().set_index('sku')['cluster']
weekly_df['cluster'] = weekly_df['sku'].map(cluster_map)

# 4. Generate Time-Series Features
def create_ts_features(group):
    # Lags (1 to 4 weeks)
    for lag in [1, 2, 3, 4]:
        group[f'qty_lag_{lag}'] = group['weekly_qty'].shift(lag)
    
    # Rolling Means
    group['rolling_mean_4'] = group['weekly_qty'].shift(1).rolling(window=4, min_periods=1).mean()
    group['rolling_mean_8'] = group['weekly_qty'].shift(1).rolling(window=8, min_periods=1).mean()
    
    # Price difference
    group['price_diff'] = group['avg_price'].diff()
    return group

weekly_df = weekly_df.sort_values(['sku', 'date'])
weekly_df = weekly_df.groupby('sku', group_keys=False).apply(create_ts_features)

# Calendar features
weekly_df['month'] = weekly_df['date'].dt.month
weekly_df['weekofyear'] = weekly_df['date'].dt.isocalendar().week.astype(int)

# Drop NaN akibat lagging
weekly_df.dropna(inplace=True)
weekly_df = weekly_df.reset_index(drop=True)

C:\Users\Asus\AppData\Local\Temp\ipykernel_28708\3339140652.py:57: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly_df = weekly_df.groupby('sku', group_keys=False).apply(create_ts_features)


In [85]:
# Cell 5: Validation & LightGBM Modeling
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit

# Sort by date for time-series split
weekly_df = weekly_df.sort_values('date').reset_index(drop=True)

# Define Features and Target
TARGET = 'weekly_qty'
FEATURES = [c for c in weekly_df.columns if c not in ['sku', 'date', TARGET]]

# Convert Categorical types for LightGBM
weekly_df['cluster'] = weekly_df['cluster'].astype('category')

# Validasi: Rolling-origin / Time-series CV
tscv = TimeSeriesSplit(n_splits=5)
fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(weekly_df)):
    train_data = weekly_df.iloc[train_idx]
    val_data = weekly_df.iloc[val_idx]
    
    X_train, y_train = train_data[FEATURES], train_data[TARGET]
    X_val, y_val = val_data[FEATURES], val_data[TARGET]
    
    train_set = lgb.Dataset(X_train, y_train, categorical_feature=['cluster'])
    val_set = lgb.Dataset(X_val, y_val, categorical_feature=['cluster'], reference=train_set)
    
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'learning_rate': 0.05,
        'num_leaves': 31,
        'verbose': -1,
        'random_state': 42
    }
    
    model = lgb.train(
        params,
        train_set,
        valid_sets=[train_set, val_set],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    preds = model.predict(X_val)
    # Cap predictions to 0 (no negative sales)
    preds = np.clip(preds, 0, None)
    
    mae = mean_absolute_error(y_val, preds)
    # MAPE bisa infinity jika y_val = 0, tambahkan epsilon
    mape = mean_absolute_percentage_error(y_val + 1e-5, preds)
    
    fold_metrics.append({'fold': fold, 'mae': mae, 'mape': mape})
    print(f"Fold {fold} | MAE: {mae:.2f} | MAPE: {mape:.2f}")

print("\nAverage Validation Metrics:")
print(pd.DataFrame(fold_metrics).mean())

Fold 0 | MAE: 0.08 | MAPE: 2078.53
Fold 1 | MAE: 0.07 | MAPE: 1521.90
Fold 2 | MAE: 0.10 | MAPE: 1554.35
Fold 3 | MAE: 0.13 | MAPE: 1957.64
Fold 4 | MAE: 0.07 | MAPE: 1197.69

Average Validation Metrics:
fold       2.000000
mae        0.089903
mape    1662.023100
dtype: float64


In [86]:
# Cell 6: Hyperparameter Tuning with Optuna

def objective(trial):
    # Holdout validation (last 20% of time) for fast tuning
    split_idx = int(len(weekly_df) * 0.8)
    train_df = weekly_df.iloc[:split_idx]
    val_df = weekly_df.iloc[split_idx:]
    
    train_set = lgb.Dataset(train_df[FEATURES], train_df[TARGET], categorical_feature=['cluster'])
    val_set = lgb.Dataset(val_df[FEATURES], val_df[TARGET], categorical_feature=['cluster'], reference=train_set)
    
    params = {
        'objective': 'regression',
        'metric': 'mae',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'verbose': -1
    }
    
    model = lgb.train(
        params, train_set, valid_sets=[val_set], num_boost_round=500,
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    
    preds = model.predict(val_df[FEATURES])
    preds = np.clip(preds, 0, None)
    return mean_absolute_error(val_df[TARGET], preds)

# Uncomment to run optimization
# study = optuna.create_study(direction='minimize')
# study.optimize(objective, n_trials=20)
# print('Best params:', study.best_params)

In [87]:
# 1. Definisi Parameter Terbaik dari Optuna (Trial 17)
best_params = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'learning_rate': 0.041917126708025984,
    'num_leaves': 58,
    'max_depth': 9,
    'feature_fraction': 0.6615160679226588,
    'bagging_fraction': 0.9998100903733517,
    'bagging_freq': 3,
    'lambda_l1': 4.298998024660962,
    'lambda_l2': 0,
    'random_state': 42,
    'verbose': -1
}

print("Mempersiapkan dataset final...")
# Menggunakan seluruh dataset weekly_df untuk model final
final_train_set = lgb.Dataset(
    weekly_df[FEATURES], 
    label=weekly_df[TARGET], 
    categorical_feature=['cluster']
)

# 2. Melatih Model Final
# Karena kita tidak menggunakan early_stopping di sini (karena tidak ada set validasi terpisah),
# kita menggunakan num_boost_round yang rasional (misal: 300-500 iterasi).
print("Melatih model LightGBM dengan parameter optimal...")
final_model = lgb.train(
    best_params,
    final_train_set,
    num_boost_round=400 
)


# 3. Sanity Check (Uji Coba Prediksi Agregasi Bulanan)
print("\n--- Sanity Check: Forecast 1 Bulan Terakhir (Aggregated) ---")

# Ambil 3 SKU paling laris untuk dicoba
top_3_skus = weekly_df.groupby('sku')[TARGET].sum().nlargest(3).index.tolist()

# Ambil 4 tanggal terakhir (representasi 4 minggu / 1 bulan terakhir)
# Karena data sekarang sudah beruntun, ini dijamin mengambil blok 4 minggu terakhir secara solid
last_4_weeks = sorted(weekly_df['date'].unique())[-4:]

# Filter dataset
sample_data = weekly_df[
    (weekly_df['sku'].isin(top_3_skus)) & 
    (weekly_df['date'].isin(last_4_weeks))
].copy()

# Lakukan prediksi mingguan
sample_preds = final_model.predict(sample_data[FEATURES])
sample_data['predicted_weekly_qty'] = np.clip(sample_preds, 0, None)

# === AGREGASI BULANAN ===
# Konversi tanggal mingguan ke format Bulan (Misal: 2026-04)
sample_data['month_year'] = sample_data['date'].dt.to_period('M').astype(str)

monthly_forecast = sample_data.groupby(['sku', 'month_year']).agg(
    total_actual_qty=(TARGET, 'sum'),            # Total aktual selama bulan itu
    total_predicted_qty=('predicted_weekly_qty', 'sum') # Total forecast selama bulan itu
).reset_index()

# Bulatkan hasil prediksi bulanan menjadi bilangan bulat (karena barang fisik tidak bisa desimal)
monthly_forecast['total_predicted_qty'] = monthly_forecast['total_predicted_qty'].round(0).astype(int)
monthly_forecast['total_actual_qty'] = monthly_forecast['total_actual_qty'].astype(int)

# Tampilkan Hasil
print(monthly_forecast.to_string(index=False))

Mempersiapkan dataset final...
Melatih model LightGBM dengan parameter optimal...

--- Sanity Check: Forecast 1 Bulan Terakhir (Aggregated) ---
                      sku month_year  total_actual_qty  total_predicted_qty
SEQUIN_BURGUNDY_1-2 Tahun    2026-04                 4                    5
SEQUIN_BURGUNDY_1-2 Tahun    2026-05                 0                    0
SEQUIN_BURGUNDY_3-4 Tahun    2026-04                 6                    6
SEQUIN_BURGUNDY_3-4 Tahun    2026-05                 0                    1
SEQUIN_BURGUNDY_5-6 Tahun    2026-04                 5                    3
SEQUIN_BURGUNDY_5-6 Tahun    2026-05                 2                    1


In [88]:
# Cell 9: True Future Forecasting (Recursive) untuk SKU Beragam

print("\n--- FUTURE FORECAST: 1 Bulan Ke Depan ---")

# 1. Pilih 3 SKU secara acak agar hasilnya beragam (tidak cuma Sequin)
# Anda bisa mengganti ini dengan list manual: sample_skus = ['SKU_A', 'SKU_B', 'SKU_C']
np.random.seed(42) # Kunci seed agar hasilnya konsisten saat di-run ulang
sample_skus = np.random.choice(weekly_df['sku'].unique(), 3, replace=False)

# 2. Ambil data historis terakhir untuk setiap SKU sebagai "Batu Pijakan"
last_known_data = weekly_df[weekly_df['sku'].isin(sample_skus)].sort_values('date').groupby('sku').tail(1).copy()

future_predictions = []
STEPS = 4 # Forecast 4 minggu ke depan (1 bulan)

for step in range(1, STEPS + 1):
    # Buat dataframe untuk minggu depan
    future_step = last_known_data.copy()
    
    # Update Tanggal (tambah 7 hari)
    future_step['date'] = future_step['date'] + pd.Timedelta(days=7)
    future_step['month'] = future_step['date'].dt.month
    future_step['weekofyear'] = future_step['date'].dt.isocalendar().week.astype(int)
    
    # Update Lags: Bergeser satu langkah
    future_step['qty_lag_4'] = future_step['qty_lag_3']
    future_step['qty_lag_3'] = future_step['qty_lag_2']
    future_step['qty_lag_2'] = future_step['qty_lag_1']
    # lag_1 diisi dengan Qty aktual minggu lalu (atau hasil prediksi jika langkah > 1)
    future_step['qty_lag_1'] = future_step['weekly_qty'] 
    
    # Prediksi menggunakan LightGBM
    preds = final_model.predict(future_step[FEATURES])
    preds = np.clip(preds, 0, None)
    
    # Simpan hasil prediksi menjadi "weekly_qty" agar bisa dipakai sebagai lag_1 di putaran (step) berikutnya
    future_step['weekly_qty'] = preds 
    
    # Simpan baris ke daftar hasil
    future_predictions.append(future_step)
    
    # Jadikan future_step sebagai last_known_data untuk iterasi berikutnya
    last_known_data = future_step.copy()

# 3. Gabungkan semua prediksi masa depan
future_df = pd.concat(future_predictions)

# 4. Agregasi Bulanan (Sama seperti sebelumnya)
future_df['month_year'] = future_df['date'].dt.to_period('M').astype(str)

monthly_future = future_df.groupby(['sku', 'month_year']).agg(
    total_forecast_qty=('weekly_qty', 'sum') # Jumlahkan prediksi 4 minggu
).reset_index()

# Bulatkan hasil
monthly_future['total_forecast_qty'] = monthly_future['total_forecast_qty'].round(0).astype(int)

# Format tabel agar enak dibaca
print(monthly_future.to_string(index=False))


--- FUTURE FORECAST: 1 Bulan Ke Depan ---
                          sku month_year  total_forecast_qty
    AMEERA_LAVENDER_7-8 Tahun    2026-05                   0
    AMEERA_LAVENDER_7-8 Tahun    2026-06                   0
DOT KERUT_BIRU MUDA_3-4 Tahun    2026-05                   0
DOT KERUT_BIRU MUDA_3-4 Tahun    2026-06                   0
JASMINE_HIJAU BOTOL_5-6 Tahun    2026-05                   0
JASMINE_HIJAU BOTOL_5-6 Tahun    2026-06                   0


In [89]:
# Cell 10: Filter Fast Cluster, Retrain, & Forecast Mei 2026

print("--- 1. FILTERING CLUSTER FAST ---")
# Menentukan otomatis cluster mana yang 'Fast' (berdasarkan rata-rata total penjualan tertinggi)
# sku_features diambil dari hasil Cell 3 sebelumnya
cluster_means = sku_features.groupby('cluster')['total_qty'].mean()
fast_cluster = cluster_means.idxmax()
slow_cluster = cluster_means.idxmin()

print(f"Rata-rata Total Qty Historis:\n{cluster_means.to_string()}")
print(f"\n=> Mengeliminasi Cluster {slow_cluster} (Slow). Hanya menggunakan Cluster {fast_cluster} (Fast).")

# Ekstrak dataframe khusus Fast SKU
weekly_df_fast = weekly_df[weekly_df['cluster'] == fast_cluster].copy()
total_fast_sku = weekly_df_fast['sku'].nunique()
print(f"=> Sisa SKU untuk diforecast: {total_fast_sku} SKU.\n")


print("--- 2. MELATIH ULANG MODEL KHUSUS FAST SKU ---")
# Kolom 'cluster' kita keluarkan dari FEATURES karena nilainya sekarang seragam semua
TARGET = 'weekly_qty'
FEATURES = [c for c in weekly_df_fast.columns if c not in ['sku', 'date', TARGET, 'cluster', 'month_year', 'predicted_weekly_qty']]

# Siapkan Dataset LightGBM tanpa categorical_feature cluster
fast_train_set = lgb.Dataset(weekly_df_fast[FEATURES], label=weekly_df_fast[TARGET])

fast_model = lgb.train(best_params, fast_train_set, num_boost_round=400)
print("=> Model Expert Fast-SKU berhasil dilatih!\n")


print("--- 3. FORECAST REKURSIF BULAN MEI 2026 ---")
# Ambil data aktual minggu TERAKHIR bulan April sebagai batu pijakan
last_known_data = weekly_df_fast.sort_values('date').groupby('sku').tail(1).copy()

future_predictions = []
STEPS = 4 # Kita forecast 4 minggu ke depan untuk mengisi penuh bulan Mei

for step in range(1, STEPS + 1):
    future_step = last_known_data.copy()
    
    # Geser kalender 7 hari ke depan
    future_step['date'] = future_step['date'] + pd.Timedelta(days=7)
    future_step['month'] = future_step['date'].dt.month
    future_step['weekofyear'] = future_step['date'].dt.isocalendar().week.astype(int)
    
    # Perbarui Lags secara kronologis
    future_step['qty_lag_4'] = future_step['qty_lag_3']
    future_step['qty_lag_3'] = future_step['qty_lag_2']
    future_step['qty_lag_2'] = future_step['qty_lag_1']
    
    # lag_1 selalu diisi dengan "weekly_qty" dari baris sebelumnya. 
    # (Step 1: Aktual akhir April. Step 2+: Hasil prediksi minggu 1 Mei)
    future_step['qty_lag_1'] = future_step['weekly_qty'] 
    
    # Update rolling mean secara dinamis menggunakan nilai lag yang baru
    future_step['rolling_mean_4'] = (future_step['qty_lag_1'] + future_step['qty_lag_2'] + 
                                     future_step['qty_lag_3'] + future_step['qty_lag_4']) / 4
    
    # Eksekusi Prediksi
    preds = fast_model.predict(future_step[FEATURES])
    future_step['weekly_qty'] = np.clip(preds, 0, None)
    
    # Simpan hasil minggu ini
    future_predictions.append(future_step)
    
    # Jadikan hasil minggu ini sebagai pijakan untuk iterasi putaran berikutnya
    last_known_data = future_step.copy()

# Gabungkan 4 minggu hasil prediksi
future_df = pd.concat(future_predictions)

# Lakukan Agregasi Bulanan (Mei 2026)
future_df['month_year'] = future_df['date'].dt.to_period('M').astype(str)
monthly_future = future_df.groupby(['sku', 'month_year']).agg(
    total_forecast_qty=('weekly_qty', 'sum')
).reset_index()

# Bulatkan hasil prediksi menjadi bilangan bulat
monthly_future['total_forecast_qty'] = monthly_future['total_forecast_qty'].round(0).astype(int)

# Tampilkan 10 sampel SKU teratas
print("=> Hasil Forecast Agregasi Bulan Mei 2026 (Menampilkan 10 SKU):")
print(monthly_future.head(10).to_string(index=False))

--- 1. FILTERING CLUSTER FAST ---
Rata-rata Total Qty Historis:
cluster
0     8.803383
1    94.633803

=> Mengeliminasi Cluster 0 (Slow). Hanya menggunakan Cluster 1 (Fast).
=> Sisa SKU untuk diforecast: 71 SKU.

--- 2. MELATIH ULANG MODEL KHUSUS FAST SKU ---
=> Model Expert Fast-SKU berhasil dilatih!

--- 3. FORECAST REKURSIF BULAN MEI 2026 ---
=> Hasil Forecast Agregasi Bulan Mei 2026 (Menampilkan 10 SKU):
                                   sku month_year  total_forecast_qty
AERA LONG DRESS_TERRACOTTA_11-12 Tahun    2026-05                   1
AERA LONG DRESS_TERRACOTTA_11-12 Tahun    2026-06                   0
              AERA_BIRU MUDA_5-6 Tahun    2026-05                   1
              AERA_BIRU MUDA_5-6 Tahun    2026-06                   0
               AERA_BURGUNDY_1-2 Tahun    2026-05                   2
               AERA_BURGUNDY_1-2 Tahun    2026-06                   1
             AERA_BURGUNDY_11-12 Tahun    2026-05                   1
             AERA_BURGUNDY_1